<a href="https://colab.research.google.com/github/nataliehany/FlyRank-ML-Internship/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nataliehany/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal verdicts

**CTR vs position — CONFIRMED.**  
CTR differs meaningfully across ranking-position buckets. Median CTR is 0.16 for positions 4–10, 0.10 for positions 11–20, and 0.00 for positions 21+. The 1–3 bucket is more skewed, with a 0.00 median but a higher mean. Overall, position is useful context when deciding whether CTR looks weak.

**Impression volume — CONFIRMED.**  
Higher-volume content represents a more substantial observed opportunity. The highest-volume quartile has a median of 9,579.5 impressions and 22 clicks, while the lowest-volume quartile has only 10 median impressions and 0 median clicks. I therefore use impression volume to prioritize which CTR opportunities deserve attention first.

### Baseline rule

My baseline identifies content that already has meaningful search visibility and a reasonable ranking position, but has weak CTR for that visibility.

A page becomes a candidate when:

- `impressions_90d >= 732`
- `avg_position <= 20`
- `ctr < 0.10`

The score increases with impression volume and gives additional priority to pages already ranking in the top 10.

**Reason code:** `LOW_CTR_VISIBLE`

**Action label:** `IMPROVE_SERP_CTR`

This is a decision-support rule, not a claim that every selected page needs an edit. The ranked queue identifies pages worth reviewing first.

In [ ]:
from pathlib import Path

repo = Path("/content/FlyRank-ML-Internship")

print("Repository exists:", repo.exists())
print("Data folder exists:", (repo / "data/raw").exists())

if (repo / "data/raw").exists():
    print("\nFiles in data/raw:")
    for f in (repo / "data/raw").iterdir():
        print("-", f.name)

Repository exists: True
Data folder exists: True

Files in data/raw:
- content_refresh_anonymized.csv


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd

# Find the starter data whether Colab opened from the repo or another directory
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in possible_paths if p.exists()), None)

if data_path is None:
    print("File not found in the expected locations.")
else:
    df = pd.read_csv(data_path)

    print("File:", data_path)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nColumn names:")
    for col in df.columns:
        print("-", col)

    print("\nData types:")
    print(df.dtypes)

    print("\nPreview:")
    display(df.head())

File: /content/FlyRank-ML-Internship/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Column names:
- content_id
- client_id
- search_volume
- competition
- competition_level
- cpc
- content_type
- main_intent
- word_count
- char_count
- provider_used
- model_used
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- age_tier
- age_tier_order
- days_since_last_update
- freshness_tier
- word_count_tier
- char_count_tier
- ctr
- avg_position
- engagement_rate
- scroll_rate
- ai_traffic_pct
- impression_tier
- position_tier
- trend_direction
- trend_pct

Data types:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competiti

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:
import numpy as np

# ============================================================
# SIGNAL 1 — CTR VS POSITION
# ============================================================

signal1 = df[
    ["avg_position", "ctr", "impressions_90d"]
].dropna().copy()

# Require some observed search exposure so CTR is meaningful
signal1 = signal1[
    signal1["impressions_90d"] > 0
].copy()

signal1["position_bucket"] = pd.cut(
    signal1["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=[
        "1-3",
        "4-10",
        "11-20",
        "21+"
    ],
    include_lowest=True
)

position_check = (
    signal1
    .groupby(
        "position_bucket",
        observed=True
    )
    .agg(
        n=("ctr", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — CTR VS POSITION")
display(position_check)


# ============================================================
# SIGNAL 2 — IMPRESSION VOLUME
# ============================================================

signal2 = df[
    ["impressions_90d", "clicks_90d", "avg_position", "ctr"]
].dropna().copy()

signal2["volume_bucket"] = pd.qcut(
    signal2["impressions_90d"],
    q=4,
    duplicates="drop"
)

volume_check = (
    signal2
    .groupby(
        "volume_bucket",
        observed=True
    )
    .agg(
        n=("impressions_90d", "size"),
        median_impressions=("impressions_90d", "median"),
        median_clicks=("clicks_90d", "median"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — IMPRESSION VOLUME")
display(volume_check)

SIGNAL 1 — CTR VS POSITION


,position_bucket,n,median_ctr,mean_ctr
0,1-3,2346,0.00,1.472869
1,4-10,11842,0.16,0.651045
2,11-20,7273,0.10,0.323443
3,21+,8539,0.00,0.211333



SIGNAL 2 — IMPRESSION VOLUME


,volume_bucket,n,median_impressions,median_clicks,median_ctr,median_position
0,"(0.999, 81.0]",7503,10.0,0.0,0.00,7.3
1,"(81.0, 731.0]",7499,300.0,0.0,0.00,15.7
2,"(731.0, 3615.25]",7498,1616.0,2.0,0.13,12.9
3,"(3615.25, 517715.0]",7500,9579.5,22.0,0.21,8.6


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I apply the single baseline rule to the observed 90-day search metrics. Eligible pages have at least 732 impressions, average position of 20 or better, and CTR below 0.10.

The score combines observed impression volume with a small top-10 ranking bonus. Higher scores mean higher review priority. Every selected row receives the same reason code and action because this baseline intentionally encodes one rule only.

The queue is ranked from highest to lowest baseline score and written to `work/outputs/baseline_action_score.csv`.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

# Work only with the fields needed by this baseline.
queue = df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position"
    ]
].copy()

# Single eligibility rule.
eligible = (
    (queue["impressions_90d"] >= 732) &
    (queue["avg_position"] <= 20) &
    (queue["ctr"] < 0.10)
)

queue = queue.loc[eligible].copy()

# Score:
# - log1p keeps very large impression counts from dominating completely
# - top-10 pages receive a small priority bonus
queue["baseline_score"] = (
    np.log1p(queue["impressions_90d"])
    + np.where(queue["avg_position"] <= 10, 1.0, 0.0)
)

# ONE reason code and ONE action label.
queue["reason_code"] = "LOW_CTR_VISIBLE"
queue["action"] = "IMPROVE_SERP_CTR"

# Highest-priority pages first.
queue = (
    queue
    .sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(1, len(queue) + 1)

# Put rank first.
queue = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position"
    ]
]

# Write the required CSV.
output_dir = Path("/content/FlyRank-ML-Internship/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Eligible rows:", len(queue))
print("CSV written to:", output_path)

print("\nTOP 20 BASELINE ACTIONS")
display(queue.head(20))

Eligible rows: 2428
CSV written to: /content/FlyRank-ML-Internship/work/outputs/baseline_action_score.csv

TOP 20 BASELINE ACTIONS


,rank,content_id,client_id,baseline_score,reason_code,action,impressions_90d,clicks_90d,ctr,avg_position
0,1,content_36ff89c8214e,client_19581e27de,13.595063,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,295097,154,0.05,7.3
1,2,content_8451fc6f034d,client_d029fa3a95,13.514090,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,272144,75,0.03,2.3
2,3,content_c84a0ab98e90,client_f369cb89fc,13.316146,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,223271,70,0.03,7.8
3,4,content_c8e9d6ab9013,client_19581e27de,13.248552,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,208678,0,0.00,9.7
4,5,content_91652435f57a,client_19581e27de,12.980370,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,159590,100,0.06,7.8
5,6,content_e12868d1f396,client_4e07408562,12.916475,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,149712,104,0.07,2.9
6,7,content_97a86caf3a3d,client_19581e27de,12.902742,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,147670,97,0.07,6.4
7,8,content_453722754fea,client_f369cb89fc,12.849969,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,140079,16,0.01,7.6
8,9,content_c1fe78bc4e37,client_19581e27de,12.806013,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,134055,43,0.03,7.5
9,10,content_4a6607efcb46,client_6208ef0f77,12.760324,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,128068,17,0.01,2.2


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

I reviewed the top 20 as decision-support candidates rather than assuming the rule is always correct. All 20 receive the same action, `IMPROVE_SERP_CTR`, and reason code, `LOW_CTR_VISIBLE`, because this baseline intentionally implements one rule.

1. **Rank 1 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 295,097 impressions, CTR 0.05, position 7.3. **Confidence: high** because visibility is very large and the page already ranks on page one. **Could be wrong if** the impressions come from queries with naturally low click intent or SERP features absorb clicks.

2. **Rank 2 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 272,144 impressions, CTR 0.03, position 2.3. **Confidence: high** because the page ranks extremely well but measured CTR remains low. **Could be wrong if** the query mix is dominated by zero-click searches or the aggregate position hides substantial query-level variation.

3. **Rank 3 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 223,271 impressions, CTR 0.03, position 7.8. **Confidence: high** because there is substantial visibility with weak measured CTR. **Could be wrong if** most impressions come from low-intent or poorly matched queries.

4. **Rank 4 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 208,678 impressions, 0 clicks, CTR 0.00, position 9.7. **Confidence: high, but requires validation** because zero clicks at this volume is unusually weak. **Could be wrong if** the source data are incomplete, tracking is missing, or impressions and clicks are not comparable for this row.

5. **Rank 5 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 159,590 impressions, CTR 0.06, position 7.8. **Confidence: high** because it has strong observed exposure and page-one ranking. **Could be wrong if** the search-result format naturally suppresses organic clicks.

6. **Rank 6 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 149,712 impressions, CTR 0.07, position 2.9. **Confidence: high** because a top-three average position makes the low CTR worth reviewing. **Could be wrong if** branded/non-branded query mix or SERP features explain the observed CTR.

7. **Rank 7 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 147,670 impressions, CTR 0.07, position 6.4. **Confidence: high** because the page has substantial exposure and a strong average position. **Could be wrong if** query intent makes the measured CTR normal for this page.

8. **Rank 8 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 140,079 impressions, CTR 0.01, position 7.6. **Confidence: high** because CTR is extremely low despite page-one visibility. **Could be wrong if** the page appears for many broad queries where it is not the preferred result.

9. **Rank 9 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 134,055 impressions, CTR 0.03, position 7.5. **Confidence: high** because the combination of large visibility and low CTR creates a clear review opportunity. **Could be wrong if** query-level performance differs substantially from these aggregates.

10. **Rank 10 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 128,068 impressions, CTR 0.01, position 2.2. **Confidence: high** because it ranks near the top while measured CTR is extremely low. **Could be wrong if** zero-click SERP behavior or tracking limitations explain the low click count.

11. **Rank 11 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 127,952 impressions, CTR 0.07, position 7.4. **Confidence: high** because it has substantial visibility at a page-one position. **Could be wrong if** 0.07 is appropriate for its particular query and SERP mix.

12. **Rank 12 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 123,469 impressions, CTR 0.03, position 8.0. **Confidence: high** because weak CTR occurs alongside large observed visibility. **Could be wrong if** impressions are concentrated in queries with weak click intent.

13. **Rank 13 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 119,217 impressions, CTR 0.02, position 7.0. **Confidence: high** because the page ranks reasonably well but receives relatively few measured clicks. **Could be wrong if** SERP features satisfy the search without a click.

14. **Rank 14 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 112,434 impressions, CTR 0.01, position 7.2. **Confidence: high** because the CTR is extremely low relative to its observed exposure. **Could be wrong if** the page is shown for broad or weakly relevant queries.

15. **Rank 15 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 111,222 impressions, CTR 0.09, position 7.4. **Confidence: medium** because it barely passes the CTR threshold. **Could be wrong if** 0.09 is normal for its query mix; this is a threshold-sensitive selection.

16. **Rank 16 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 106,652 impressions, CTR 0.09, position 4.8. **Confidence: medium** because the page has strong visibility but sits just below the CTR cutoff. **Could be wrong if** its expected CTR is close to 0.09 for the searches where it appears.

17. **Rank 17 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 105,643 impressions, CTR 0.09, position 5.7. **Confidence: medium** because it is another borderline CTR selection. **Could be wrong if** small threshold changes would remove it or its query mix normally produces this CTR.

18. **Rank 18 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 99,013 impressions, CTR 0.03, position 6.4. **Confidence: high** because low CTR persists despite strong visibility and page-one ranking. **Could be wrong if** broad query matching inflates impressions without representing a realistic click opportunity.

19. **Rank 19 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 97,235 impressions, CTR 0.07, position 6.5. **Confidence: high** because observed visibility is large and ranking is strong enough to justify review. **Could be wrong if** its observed CTR is normal for the underlying query intent.

20. **Rank 20 — IMPROVE_SERP_CTR / LOW_CTR_VISIBLE.** 90,991 impressions, CTR 0.08, position 4.5. **Confidence: medium** because it has strong ranking and volume but is relatively close to the CTR threshold. **Could be wrong if** 0.08 is an expected CTR for its query/feature mix.

### Review conclusion

The strongest picks are pages combining very high impressions, page-one positions, and extremely low CTR. The less convincing picks are those close to the 0.10 CTR threshold. The review therefore supports using the queue for prioritization, but not treating the baseline score as proof that a page definitely requires a CTR intervention.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the exact top 20 selected by the baseline.
top20 = queue.head(20).copy()

print("Rows reviewed:", len(top20))

display(
    top20[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position"
        ]
    ]
)


Rows reviewed: 20


,rank,content_id,baseline_score,reason_code,action,impressions_90d,clicks_90d,ctr,avg_position
0,1,content_36ff89c8214e,13.595063,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,295097,154,0.05,7.3
1,2,content_8451fc6f034d,13.514090,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,272144,75,0.03,2.3
2,3,content_c84a0ab98e90,13.316146,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,223271,70,0.03,7.8
3,4,content_c8e9d6ab9013,13.248552,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,208678,0,0.00,9.7
4,5,content_91652435f57a,12.980370,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,159590,100,0.06,7.8
5,6,content_e12868d1f396,12.916475,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,149712,104,0.07,2.9
6,7,content_97a86caf3a3d,12.902742,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,147670,97,0.07,6.4
7,8,content_453722754fea,12.849969,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,140079,16,0.01,7.6
8,9,content_c1fe78bc4e37,12.806013,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,134055,43,0.03,7.5
9,10,content_4a6607efcb46,12.760324,LOW_CTR_VISIBLE,IMPROVE_SERP_CTR,128068,17,0.01,2.2


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The weakest top-20 selections are ranks 15, 16, 17, and 20 because their CTR values (0.09, 0.09, 0.09, and 0.08) are close to the rule's 0.10 cutoff. Small changes to the threshold or differences in query intent could remove these pages from the queue.

Rank 4 also deserves a manual data-quality check. It has 208,678 impressions but zero clicks while averaging position 9.7. This may represent a genuine CTR problem, but incomplete click tracking or unusual query/SERP behavior could also explain it.

This shows an important limitation of the baseline: the rule uses aggregate page-level metrics. It does not know the query mix, SERP layout, expected CTR for each query, or whether tracking is complete. Therefore, the score is a prioritization signal for review, not proof that an intervention is correct.

### Leakage check

The baseline uses only observed search-performance inputs available at the scoring decision moment:

- `impressions_90d`
- `ctr`
- `avg_position`

`clicks_90d` is displayed for interpretation but is not independently used by the scoring formula.

I did **not** use future-window outcomes, labels, product-generated flags, `trend_direction`, or any target-derived field to select or score rows. The reason code and action label are outputs created by this rule, not inputs.

**Leakage verdict: PASS.** The baseline is based only on contemporaneously observed search signals and contains no future-window or label-derived input.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak-pick and leakage checks.

weak_picks = top20[top20["ctr"] >= 0.08].copy()

print("WEAK / THRESHOLD-SENSITIVE TOP-20 PICKS")
display(
    weak_picks[
        [
            "rank",
            "content_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "baseline_score"
        ]
    ]
)

# Explicitly document the fields used to select and score the queue.
selection_inputs = {
    "impressions_90d",
    "ctr",
    "avg_position"
}

score_inputs = {
    "impressions_90d",
    "avg_position"
}

prohibited_inputs = {
    "trend_direction",
    "trend_pct",
    "freshness_tier",
    "impression_tier",
    "position_tier"
}

used_inputs = selection_inputs | score_inputs
leaked_inputs = used_inputs.intersection(prohibited_inputs)

print("\nLEAKAGE CHECK")
print("Selection inputs:", sorted(selection_inputs))
print("Score inputs:", sorted(score_inputs))
print("Prohibited/product-derived inputs used:", sorted(leaked_inputs))
print("Future-window inputs used: []")
print("Label-derived inputs used: []")

assert len(leaked_inputs) == 0

print("\nLeakage verdict: PASS")

WEAK / THRESHOLD-SENSITIVE TOP-20 PICKS


,rank,content_id,impressions_90d,ctr,avg_position,baseline_score
14,15,content_647e177596e9,111222,0.09,7.4,12.619292
15,16,content_42d423551e2c,106652,0.09,4.8,12.577336
16,17,content_76e77629e9f1,105643,0.09,5.7,12.567830
19,20,content_87dfc063bf4e,90991,0.08,4.5,12.418527



LEAKAGE CHECK
Selection inputs: ['avg_position', 'ctr', 'impressions_90d']
Score inputs: ['avg_position', 'impressions_90d']
Prohibited/product-derived inputs used: []
Future-window inputs used: []
Label-derived inputs used: []

Leakage verdict: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.